## Baseline: Fine-Tuning Naive

In [34]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
from utils import load_classifier
import torch.nn.functional as F
import torch
from train import train_classifier, evaluate_classifier,evaluate_task_incremental
from dataloaders import SequentialCIFAR10
from torch import nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


In [36]:
seq_cifar = SequentialCIFAR10(data_root="./data", batch_size=32, buffer_size=200)

In [ ]:
def train_and_test_task(task_nr, epochs = 2, criterion = nn.CrossEntropyLoss()):
    print(f"---------------- Training task {task_nr} -------------------------")
    train_dataloader, val_dataloader = seq_cifar.get_task_il_train_val_loaders(task_id=task_nr)
    model = load_classifier('checkpoints/task_0_pretrain', device=device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    train_classifier(
        model,
        train_dataloader,
        val_dataloader,
        optimizer,
        criterion,
        num_epochs=epochs,
        device=device,
    )
    test_loaders = seq_cifar.get_task_il_test_loaders(task_nr)
    
    task_results = evaluate_task_incremental(model, task_nr, test_loaders, criterion, device=device)
    for eval_task, results in task_results.items():
        print(f"Task {eval_task} - Test Loss: {results['loss']:.4f}, Test Accuracy: {results['acc']:.4f}")

    #save 
    torch.save(model.state_dict(), f'checkpoints/task_{task_nr}_model.pth')

In [38]:
for task in range(0, 5):
    train_and_test_task(task, epochs=2)

---------------- Training task 0 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training:   0%|          | 0/2 [00:00<?, ?it/s]/Users/alexanderbodner/Documents/Udesa/5to/vision avanzada/tp1_continual_learning/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Training: 100%|██████████| 2/2 [00:54<00:00, 27.46s/it]


Task task_0 - Test Loss: 0.3275, Test Accuracy: 0.9070
---------------- Training task 1 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training: 100%|██████████| 2/2 [00:55<00:00, 27.93s/it]


Task task_0 - Test Loss: 0.5097, Test Accuracy: 0.8625
Task task_1 - Test Loss: 0.6715, Test Accuracy: 0.5865
---------------- Training task 2 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


...

Task task_0 - Test Loss: 0.5248, Test Accuracy: 0.8550
Task task_1 - Test Loss: 0.6755, Test Accuracy: 0.5755
Task task_2 - Test Loss: 0.6837, Test Accuracy: 0.5540
---------------- Training task 3 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training: 100%|██████████| 2/2 [00:55<00:00, 27.87s/it]


Task task_0 - Test Loss: 0.9314, Test Accuracy: 0.1345
Task task_1 - Test Loss: 0.7387, Test Accuracy: 0.4525
Task task_2 - Test Loss: 0.7283, Test Accuracy: 0.4525
Task task_3 - Test Loss: 0.6600, Test Accuracy: 0.6140
---------------- Training task 4 -------------------------
Backbone + linear head loaded successfully into reloaded_linear_probe.


Training: 100%|██████████| 2/2 [00:55<00:00, 27.90s/it]


Task task_0 - Test Loss: 0.3681, Test Accuracy: 0.9010
Task task_1 - Test Loss: 0.6982, Test Accuracy: 0.5675
Task task_2 - Test Loss: 0.7067, Test Accuracy: 0.5500
Task task_3 - Test Loss: 0.8256, Test Accuracy: 0.4015
Task task_4 - Test Loss: 0.5280, Test Accuracy: 0.7540


In [ ]:
for task in range(0, 5):
    train_and_test_task(task, epochs=10)

# EWC

In [ ]:
class EWCLoss(nn.Module):
    def __init__(self, lambda_reg=0.25):
        super().__init__()
        self.loss1 = nn.CrossEntropyLoss() 


        #setting these to 0.25 so their total starts out as 1.0. 
        self.lambda_ = nn.Parameter(lambda_reg)


    def forward(self, x, model_taskB, model_taskA):
        ewc_term_loss = 0 
        for index, param in enumerate(model_taskB.parameters()):
            fisher = param.grad ** 2
            ewc_term_loss += (fisher * (param - model_taskA.parameters()[index].detach()) ** 2).sum()
        return  self.loss1(x) + self.lambda_ * ewc_term_loss

In [ ]:
from train_ewc import train_ewc